In [2]:
import pandas as pd
import numpy as np

In [8]:
opps = pd.read_csv('../data_bronze/opportunities.csv')
sales = pd.read_csv('../data_bronze/salespeople.csv')
accounts = pd.read_csv('../data_bronze/accounts.csv')
activities = pd.read_csv('../data_bronze/activities.csv')
targets = pd.read_csv('../data_bronze/targets.csv')

# Explorer
for name, df in [('opps', opps), ('sales', sales), ('accounts', accounts), ('activities', activities), ('targets', targets)]:
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Duplicates: {df.duplicated().sum()}")
    print(df.info())


opps:
  Shape: (3300, 9)
  Columns: ['opportunity_id', 'created_date', 'closed_date', 'arr', 'status', 'salesperson_id', 'source', 'type', 'account_id']
  Duplicates: 0
<class 'pandas.DataFrame'>
RangeIndex: 3300 entries, 0 to 3299
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   opportunity_id  3300 non-null   str  
 1   created_date    3300 non-null   str  
 2   closed_date     3300 non-null   str  
 3   arr             3300 non-null   int64
 4   status          3300 non-null   str  
 5   salesperson_id  3300 non-null   str  
 6   source          3255 non-null   str  
 7   type            3300 non-null   str  
 8   account_id      3300 non-null   str  
dtypes: int64(1), str(8)
memory usage: 452.1 KB
None

sales:
  Shape: (10, 3)
  Columns: ['salesperson_id', 'name', 'salesperson_office']
  Duplicates: 0
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column       

In [10]:
import pandas as pd
import numpy as np

# ============================================================================
# CHARGEMENT
# ============================================================================
salespeople = pd.read_csv('../data_bronze/salespeople.csv')
# accounts = pd.read_csv('accounts.csv')
# activities = pd.read_csv('activities.csv')
# targets = pd.read_csv('targets.csv')

print("="*80)
print("DATA QUALITY EXPLORATION")
print("="*80)

# ============================================================================
# SALESPEOPLE.CSV
# ============================================================================
print("\n📋 SALESPEOPLE")
print(f"Shape: {salespeople.shape}")
print(f"Columns: {salespeople.columns.tolist()}")

# Check 1: Unicité de salesperson_id
duplicates = salespeople['salesperson_id'].duplicated().sum()
print(f"❌ Duplicate salesperson_id: {duplicates}")

# Check 2: Valeurs manquantes
print(f"Missing values:\n{salespeople.isnull().sum()}")

# Check 3: Valeurs distinctes pour salesperson_office
print(f"Distinct offices: {salespeople['salesperson_office'].unique()}")
print(f"Office value counts:\n{salespeople['salesperson_office'].value_counts()}")

# Check 4: Incohérences de casse ou espaces
salespeople['salesperson_office_cleaned'] = salespeople['salesperson_office'].str.strip().str.title()
if (salespeople['salesperson_office'] != salespeople['salesperson_office_cleaned']).any():
    print("⚠️  WARNING: Incohérences de casse/espaces détectées dans salesperson_office")
    print(salespeople[salespeople['salesperson_office'] != salespeople['salesperson_office_cleaned']][['salesperson_id', 'salesperson_office']])

# ============================================================================
# ACCOUNTS.CSV
# ============================================================================
print("\n📋 ACCOUNTS")
print(f"Shape: {accounts.shape}")
print(f"Columns: {accounts.columns.tolist()}")

# Check 1: Unicité de account_id
duplicates = accounts['account_id'].duplicated().sum()
print(f"❌ Duplicate account_id: {duplicates}")

# Check 2: Valeurs manquantes
print(f"Missing values:\n{accounts.isnull().sum()}")

# Check 3: Valeurs distinctes pour account_office
print(f"Distinct offices: {accounts['account_office'].unique()}")
print(f"Office value counts:\n{accounts['account_office'].value_counts()}")

# Check 4: Cohérence avec salespeople offices
sales_offices = set(salespeople['salesperson_office'].unique())
account_offices = set(accounts['account_office'].unique())
if sales_offices != account_offices:
    print(f"⚠️  WARNING: Offices différents entre salespeople et accounts")
    print(f"   In salespeople only: {sales_offices - account_offices}")
    print(f"   In accounts only: {account_offices - sales_offices}")

# ============================================================================
# ACTIVITIES.CSV
# ============================================================================
print("\n📋 ACTIVITIES")
print(f"Shape: {activities.shape}")
print(f"Columns: {activities.columns.tolist()}")

# Check 1: Unicité de activity_id
duplicates = activities['activity_id'].duplicated().sum()
print(f"❌ Duplicate activity_id: {duplicates}")

# Check 2: Valeurs manquantes
print(f"Missing values:\n{activities.isnull().sum()}")

# Check 3: Format de activity_date (même problème que opportunities ?)
print(f"Sample activity_date values:\n{activities['activity_date'].head(10)}")
print(f"Unique date formats detected:")
print(f"  Contains '-': {activities['activity_date'].astype(str).str.contains('-').sum()}")
print(f"  Contains '/': {activities['activity_date'].astype(str).str.contains('/').sum()}")

# Check 4: Types d'activités
print(f"Distinct activity_type: {activities['activity_type'].unique()}")
print(f"Activity type value counts:\n{activities['activity_type'].value_counts()}")

# Check 5: Incohérences de casse
activities['activity_type_cleaned'] = activities['activity_type'].str.strip().str.lower()
if (activities['activity_type'].str.lower() != activities['activity_type_cleaned']).any():
    print("⚠️  WARNING: Incohérences de casse dans activity_type")

# Check 6: Intégrité référentielle (opportunity_id existe dans opportunities ?)
# NOTE: tu devras charger opportunities pour faire ce test

# ============================================================================
# TARGETS.CSV
# ============================================================================
print("\n📋 TARGETS")
print(f"Shape: {targets.shape}")
print(f"Columns: {targets.columns.tolist()}")

# Check 1: Unicité de la combinaison salesperson_id + account_office
duplicates = targets.duplicated(subset=['salesperson_id', 'account_office']).sum()
print(f"❌ Duplicate (salesperson_id, account_office): {duplicates}")

# Check 2: Valeurs manquantes
print(f"Missing values:\n{targets.isnull().sum()}")

# Check 3: Distribution des targets
print(f"Target statistics:")
print(targets['quarter_target'].describe())

# Check 4: Targets négatifs ou nuls
negative_targets = (targets['quarter_target'] <= 0).sum()
print(f"❌ Negative or zero targets: {negative_targets}")

# Check 5: Cohérence des offices avec salespeople
target_offices = set(targets['account_office'].unique())
if target_offices != account_offices:
    print(f"⚠️  WARNING: Offices différents entre targets et accounts")
    print(f"   In targets only: {target_offices - account_offices}")
    print(f"   In accounts only: {account_offices - target_offices}")

# Check 6: CRUCIAL - Y a-t-il plusieurs lignes par salesperson ?
print(f"\nDistribution des targets par salesperson:")
target_per_sales = targets.groupby('salesperson_id').size()
print(f"  Min targets per salesperson: {target_per_sales.min()}")
print(f"  Max targets per salesperson: {target_per_sales.max()}")
print(f"  Median: {target_per_sales.median()}")

if target_per_sales.max() > 1:
    print(f"⚠️  INFO: Certains commerciaux ont plusieurs targets (probablement par office)")
    print(f"Sample:\n{targets[targets['salesperson_id'].isin(target_per_sales[target_per_sales > 1].index)].head(10)}")

# ============================================================================
# CROSS-TABLE CHECKS
# ============================================================================
print("\n" + "="*80)
print("CROSS-TABLE INTEGRITY CHECKS")
print("="*80)

# Tu devras charger opportunities aussi pour ces checks
opps = pd.read_csv('opportunities.csv')

# Check 1: Tous les salesperson_id dans opportunities existent dans salespeople ?
opps_sales = set(opps['salesperson_id'].dropna().unique())
master_sales = set(salespeople['salesperson_id'].unique())
orphan_sales = opps_sales - master_sales
print(f"\n❌ salesperson_id in opportunities but not in salespeople: {len(orphan_sales)}")
if orphan_sales:
    print(f"   Orphan IDs: {orphan_sales}")

# Check 2: Tous les account_id dans opportunities existent dans accounts ?
opps_accounts = set(opps['account_id'].dropna().unique())
master_accounts = set(accounts['account_id'].unique())
orphan_accounts = opps_accounts - master_accounts
print(f"❌ account_id in opportunities but not in accounts: {len(orphan_accounts)}")
if orphan_accounts:
    print(f"   Orphan IDs: {list(orphan_accounts)[:10]} (showing first 10)")

# Check 3: Tous les opportunity_id dans activities existent dans opportunities ?
activities_opps = set(activities['opportunity_id'].dropna().unique())
master_opps = set(opps['opportunity_id'].unique())
orphan_opps = activities_opps - master_opps
print(f"❌ opportunity_id in activities but not in opportunities: {len(orphan_opps)}")
if orphan_opps:
    print(f"   Orphan IDs: {list(orphan_opps)[:10]} (showing first 10)")

# Check 4: Tous les salesperson_id dans targets existent dans salespeople ?
target_sales = set(targets['salesperson_id'].unique())
orphan_target_sales = target_sales - master_sales
print(f"❌ salesperson_id in targets but not in salespeople: {len(orphan_target_sales)}")
if orphan_target_sales:
    print(f"   Orphan IDs: {orphan_target_sales}")

print("\n" + "="*80)
print("EXPLORATION COMPLETE")
print("="*80)

DATA QUALITY EXPLORATION

📋 SALESPEOPLE
Shape: (10, 3)
Columns: ['salesperson_id', 'name', 'salesperson_office']
❌ Duplicate salesperson_id: 0
Missing values:
salesperson_id        0
name                  0
salesperson_office    0
dtype: int64
Distinct offices: <ArrowStringArray>
['France', 'UK', 'US', 'Germany', 'Singapore']
Length: 5, dtype: str
Office value counts:
salesperson_office
France       3
UK           3
US           2
Germany      1
Singapore    1
Name: count, dtype: int64
⚠️  WARNING: Incohérences de casse/espaces détectées dans salesperson_office
  salesperson_id salesperson_office
3          SP004                 UK
4          SP005                 UK
5          SP006                 UK
6          SP007                 US
7          SP008                 US

📋 ACCOUNTS
Shape: (120, 3)
Columns: ['account_id', 'account_name', 'account_office']
❌ Duplicate account_id: 0
Missing values:
account_id        0
account_name      0
account_office    0
dtype: int64
Distinct office

FileNotFoundError: [Errno 2] No such file or directory: 'opportunities.csv'